# Silver — ERP Product Categories
Category / subcategory lookup for products.

`bronze.erp_px_cat_g1v2` → `silver.erp_product_category`

## Init

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType, DateType

CATALOG = "workspace"

## Read bronze table

In [ ]:
df = spark.table(f"{CATALOG}.bronze.erp_px_cat_g1v2")

## Transformations

### Trim all string columns

In [ ]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Maintenance flag to boolean

In [ ]:
df = df.withColumn(
    "maintenance",
    F.when(F.upper(col("maintenance")) == "YES", F.lit(True))
     .when(F.upper(col("maintenance")) == "NO", F.lit(False))
     .otherwise(None)
)

### Rename to business-friendly names

In [ ]:
RENAME_MAP = {
    "id": "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance_flag"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity check

In [ ]:
df.limit(10).display()

## Write silver table

In [ ]:
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{CATALOG}.silver.erp_product_category")

In [ ]:
%sql
SELECT * FROM workspace.silver.erp_product_category LIMIT 10;